# Spark Memory Management: Interview Problem Solving Guide
## Step-by-Step Frameworks with Examples

## Section 1: Core Calculation Framework
### 1.1 Memory Components Breakdown

```mermaid
graph TD
    A[Total Container Memory] --> B[Executor Heap]
    A --> C[Overhead Memory]
    B --> D[Reserved Memory]
    B --> E[Usable Memory]
    E --> F[Unified Memory]
    E --> G[User Memory]
    F --> H[Execution]
    F --> I[Storage]
```

**Key Formulas:**
1. `Overhead Memory = MAX(384MB, 10% of spark.executor.memory)`
2. `Usable Memory = Heap - 300MB (Reserved)`
3. `Unified Memory = Usable * spark.memory.fraction (default 0.6)`
4. `Storage/Execution Split = Unified * spark.memory.storageFraction (default 0.5)`

### 1.2 Interactive Calculator
**Try adjusting these values:**
```python
# Input your scenario
executor_memory_gb = 4
num_executors = 3
off_heap_enabled = False

# Calculations
overhead_mb = max(384, executor_memory_gb * 1024 * 0.1)
usable_mb = (executor_memory_gb * 1024) - 300
unified_mb = usable_mb * 0.6
storage_mb = unified_mb * 0.5

print(f"""
Configuration Analysis:
Executor Heap: {executor_memory_gb}GB
Overhead Memory: {overhead_mb/1024:.1f}GB
Usable Memory: {usable_mb/1024:.1f}GB
Storage Memory: {storage_mb/1024:.1f}GB per executor
Total Cluster Storage: {storage_mb * num_executors / 1024:.1f}GB
""")
```

**Sample Output:**
```
Configuration Analysis:
Executor Heap: 4GB
Overhead Memory: 0.4GB
Usable Memory: 3.7GB
Storage Memory: 1.1GB per executor
Total Cluster Storage: 3.3GB
```

## Section 2: Common Problem Patterns
### 2.1 YARN Memory Exceeded
**Problem Statement:**
"Your Spark job fails with 'Container killed by YARN for exceeding 8GB limit'"

**Diagnosis Steps:**
1. Check current allocation:
```python
executor_memory = 7  # GB
overhead = max(384, executor_memory * 1024 * 0.1) / 1024  # GB
total = executor_memory + overhead
print(f"Total container memory: {total:.1f}GB")
# Output: Total container memory: 7.7GB
```

2. Identify threshold violation:
```python
yarn_limit = 8  # GB
if total > yarn_limit:
    print(f"Violation: {total:.1f}GB > {yarn_limit}GB limit")
```

**Solution:**
```python
new_executor_memory = 6  # GB
new_overhead = max(384, new_executor_memory * 1024 * 0.1) / 1024
print(f"Adjusted memory: {new_executor_memory + new_overhead:.1f}GB")
# Output: Adjusted memory: 6.6GB
```

### 2.2 Shuffle Memory Planning
**Problem Statement:**
"Estimate memory needed for a 50GB shuffle with 200 partitions"

**Calculation Framework:**
```python
shuffle_size_gb = 50
partitions = 200
concurrent_tasks = 4  # --executor-cores

per_partition_mb = (shuffle_size_gb * 1024) / partitions
required_mb = per_partition_mb * concurrent_tasks

print(f"""
Shuffle Memory Requirements:
Per partition: {per_partition_mb:.1f}MB
Per executor ({concurrent_tasks} cores): {required_mb:.1f}MB
""")
```

**Validation Check:**
```python
unified_mb = (executor_memory * 1024 - 300) * 0.6
execution_mb = unified_mb * 0.5

if required_mb < execution_mb:
    print("✅ Sufficient memory")
else:
    print(f"❌ Increase execution memory to at least {required_mb/1024:.1f}GB")
```

## Section 3: Configuration Templates
### 3.1 Workload-Specific Presets
**ETL Workload Template:**
```python
etl_config = {
    "spark.executor.memory": "8g",
    "spark.memory.fraction": "0.8",  # Prioritize execution
    "spark.memory.storageFraction": "0.3",
    "spark.shuffle.spill": "true",
    "spark.sql.shuffle.partitions": "400"  # More partitions = less memory pressure
}
```

**ML Workload Template:**
```python
ml_config = {
    "spark.executor.memory": "12g",
    "spark.memory.offHeap.enabled": "true",
    "spark.memory.offHeap.size": "4g",
    "spark.executor.pyspark.memory": "2g",
    "spark.sql.execution.arrow.pyspark.enabled": "true"
}
```

### 3.2 Troubleshooting Cheat Sheet
| Symptom | Diagnosis | Solution |
|---------|-----------|----------|
| **GC Overhead** | High GC time in Spark UI | Enable off-heap: `spark.memory.offHeap.size="2g"` |
| **Cache Evictions** | Low "Storage Memory" in UI | Increase `spark.memory.storageFraction` |
| **Python OOM** | Worker crashes | Set `spark.executor.pyspark.memory="1g"` |
| **Shuffle Failures** | `FetchFailedException` | Increase `spark.sql.shuffle.partitions` |

## Section 4: Interactive Practice
### 4.1 Case Study: E-Commerce Pipeline
**Scenario:**
- Cluster: 5 nodes, 16GB each
- Workload: Daily sales aggregation (20GB shuffle) + product caching
- Issues: Slow shuffles, cache evictions

**Your Task:**
```python
# Calculate optimal configuration
nodes = 5
node_memory = 16  # GB

# Suggested parameters to compute:
executors_per_node = ?
executor_memory = ?
shuffle_partitions = ?

# Validate your solution:
total_memory = executors_per_node * nodes * (executor_memory + max(384, executor_memory*1024*0.1)/1024)
assert total_memory < nodes * node_memory * 0.9  # 10% headroom
```

### 4.2 Answer Key
**Optimal Configuration:**
```python
solution = {
    "executors_per_node": 2,  # Leave room for OS/daemons
    "executor_memory": "6g",  # 6GB heap
    "executor_overhead": "1g",  # MAX(384MB, 600MB)
    "total_executors": 10,
    "shuffle_partitions": "500",
    "config": {
        "spark.memory.fraction": "0.7",
        "spark.memory.storageFraction": "0.5"
    }
}
```

**Validation:**
```python
total_per_node = 6 + 1 = 7GB
cluster_usage = 5 nodes * 2 executors * 7GB = 70GB
cluster_capacity = 5 * 16 = 80GB
print(f"Headroom: {80-70}GB (12.5%)")  # ✅ Healthy
```